In [1]:
from langchain_openai import ChatOpenAI  
from langchain_core.messages import HumanMessage

LLM = ChatOpenAI(model="gpt-4o-mini")   # Initialize the chat model

In [2]:
for c in LLM.stream([HumanMessage("안녕? AI로 인한 인력 채용 변화에 대해 이야기해줘.")]):
    print(c.content, end='|') 

|안|녕하세요|!| AI|의| 발전|은| 인|력| 채|용| 과정|에| 많은| 변|화를| 가져|오|고| 있습니다|.| 다음|은| AI|가| 인|력| 채|용|에| 미|치는| 주요| 영향| 몇| 가지|입니다|.

|1|.| **|자동|화|된| 이|력|서| 스|크|리|닝|**|:| 많은| 기업|들이| 이|력|서| 스|크|리|닝| 과정|에서| AI|를| 사용|하여| 지원|자의| 이|력|서를| 자동|으로| 분석|하고|,| 특정| 키|워|드|나| 요구| 사항|에| 맞|는| 지원|자를| 필|터|링|합니다|.| 이를| 통해| 인|사| 담당|자|들은| 더| 효|율|적으로| 후보|자를| 선|별|할| 수| 있습니다|.

|2|.| **|인|공지|능| 기반| 평가| 도|구|**|:| AI|는| 지원|자의| 기술|,| 성|격|,| 문제| 해결| 능|력| 등을| 평가|하는| 다양한| 도|구|를| 제공|하고| 있습니다|.| 예|를| 들어|,| 온라인| 테스트| 및| 대|화|형| 인터|뷰|를| 통해| 지원|자의| 적|합|성을| 판|별|할| 수| 있습니다|.

|3|.| **|데|이터| 분석|과| 예|측|**|:| AI|는| 인|력| 채|용| 데이터| 분석|을| 통해| 과|거|의| 채|용| 사례|를| 조사|하고|,| 어떤| 특|성이| 성공|적인| 직원|과| 관련|이| 있는|지를| 식|별|할| 수| 있습니다|.| 이러한| 데이터| 기반|의| 접근| 방식|은| 더| 나|은| 채|용| 결|정을| 내|리는| 데| 도움|을| 줄| 수| 있습니다|.

|4|.| **|지원|자| 경험| 개선|**|:| AI| 챗|봇|을| 활용|하여| 지원|자|와|의| 소|통|을| 강화|하고|,| 질문|에| 대한| 즉|각|적인| 답|변|을| 제공|하는| 등| 지원|자| 경험|을| 향|상|시|킬| 수| 있습니다|.| 이는| 지원|자가| 불|편|함|을| 느|끼|지| 않|도록| 돕|고|,| 기업|의| 이미지|에도| 긍|정|적인| 영향을| 미|칠| 수| 있습니다|.

|5|.| **|편|견| 최소|화|**|:| AI|를| 사용|

In [3]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool       # @tool decorator uses langchain's tool system
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각 반환 함수
    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 LLM 답변 생성에 사용됨
    """
    Time_zone = pytz.timezone(timezone)
    now = datetime.now(Time_zone).strftime("%Y-%m-%d %H:%M:%S")

    # Return format: 'Asia/Seoul (Seoul) 현재시각 2024-06-01 15:30:00'
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} '
     
    print(location_and_local_time)
    return location_and_local_time

In [4]:
# Append the tool to a list and a dictionary
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# Bind the tools to the LLM 
LLM_with_tools = LLM.bind_tools(tools)

In [5]:
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야"),
    HumanMessage("인천은 지금 몇시야?"),
]

response = LLM_with_tools.stream(messages)

# Gather the streamed chunks into a single message
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'type': 'tool_c

In [6]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4c55ec580f', 'service_tier': 'default'}, id='lc_run--01a19ffc-c7c8-76cd-8356-7b9b6154a164', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '인천'}, 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 23, 'total_tokens': 161, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"인천"}', 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [7]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]    # Select the tool based on the name tool_call
    print(tool_call["args"])                        # Arguments for the tool function
    tool_msg = selected_tool.invoke(tool_call)      # Invoke the tool with the tool_call
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '인천'}
Asia/Seoul (인천) 현재시각 2026-11-03 14:58:35 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='인천은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4c55ec580f', 'service_tier': 'default'}, id='lc_run--01a19ffc-c7c8-76cd-8356-7b9b6154a164', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '인천'}, 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 23, 'total_tokens': 161, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"인천"}', 'id': 'call_BVhnlqrsAlbRTMGTJaFawPIz', 'index': 

In [8]:
for c in LLM_with_tools.stream(messages):
    print(c.content, end='|')

|인|천|은| 현재| |202|6|년| |11|월| |3|일| |14|시| |58|분|입니다|.||||